In [ ]:
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
import os
import time
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, classification_report
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

def high_citation_predict(title, abstract, keywords, year_cleaning, publisher):
   prompt = f"""
You are an expert in Statistics and Econometrics with extensive peer-review experience, specializing in research impact evaluation and identifying characteristics of high-impact academic papers.

Your task is to evaluate whether the following academic paper demonstrates the potential to become **highly cited** in its research field. 

---
**Guidelines for Identifying High-Citation Potential Papers**  
To guide your evaluation, consider the following widely accepted characteristics of highly cited academic work:

1. **Methodological Innovation**  
   - Proposes *new statistical/computational methods* OR *significantly improves existing ones*  
   - Solves critical bottlenecks (e.g., computational efficiency, accuracy limitations)  
   - Bridges novel methodology with theoretical justification or practical implementation  

2. **Long-term Value**  
   - Advances *foundational theories* with broad implications  
   - Demonstrates potential to *shift research paradigms* in its field  

3. **Problem Significance**  
   - Addresses *high-stakes scientific/societal challenges*  
   - Shows clear *practical utility* in real-world applications  

---
**Field and Temporal Background**  
Papers published in the late 1990s (1996--2000) emerged during a period marked by accelerated methodological innovation and the consolidation of 
computationally intensive approaches. Advances in nonparametric regression, spline smoothing, and Bayesian hierarchical modeling gained adoption in Statistics. 
In Econometrics, increased availability of panel data stimulated work on dynamic panel estimators and generalized method of moments (GMM), while simulation-based 
inference expanded the practical analytical toolkit. Machine Learning was shaped by support vector machines, boosting, and ensemble methods, which offered new paradigms 
for prediction and classification. Computational statistics benefited from developments in MCMC, enabling Bayesian applications to high-dimensional problems. Applications 
in macroeconomics, finance, and social sciences were complemented by cross-disciplinary research in bioinformatics and information retrieval, reflecting the importance of 
large heterogeneous datasets.

---
**Reference Examples for High-Citation Assessment**  
Below are several examples illustrating how to distinguish between highly cited and not highly cited papers.:

**Example 1: YES (Highly Cited)**  
**Title:** "Greedy function approximation: A gradient boosting machine"  
**Abstract:**  
"Function estimation/approximation is viewed from the perspective of numerical optimization in function space, rather than parameter space. A connection is made between stagewise additive expansions and steepest-descent minimization. A general gradient descent "boosting" paradigm is developed for additive expansions based on any fitting criterion. Specific algorithms are presented for least-squares, least absolute deviation, and Huber-M loss functions for regression, and multiclass logistic likelihood for classification. Special enhancements are derived for the particular case where the individual additive components are regression trees, and tools for interpreting such "TreeBoost" models are presented. Gradient boosting of regression trees produces competitive, highly robust, interpretable procedures for both regression and classification, especially appropriate for ruining less than clean data. Connections between this approach and the boosting methods of Freund and Shapire and Friedman, Hastie and Tibshirani are discussed."
**Keywords:** 'function estimation', 'boosting', 'decision trees', 'robust nonparametric regression'
**publisher:** ANNALS OF STATISTICS
**→ Judgment:** YES

---
**Example 2: YES (Highly Cited)**
**Title:** "Regularization and variable selection via the elastic net"  
**Abstract:**  
"We propose the elastic net, a new regularization and variable selection method. Real world data and a simulation study show that the elastic net often outperforms the lasso, while enjoying a similar sparsity of representation. In addition, the elastic net encourages a grouping effect, where strongly correlated predictors tend to be in or out of the model together. The elastic net is particularly useful when the number of predictors (p) is much bigger than the number of observations (n). By contrast, the lasso is not a very satisfactory variable selection method in the p>n case. An algorithm called LARS-EN is proposed for computing elastic net regularization paths efficiently, much like algorithm LARS does for the lasso."
**Keywords:** 'grouping effect', 'LARS algorithm', 'Lasso', 'penalization', 'p >> n problem', 'variable selection'
**publisher:** JOURNAL OF THE ROYAL STATISTICAL SOCIETY SERIES B-STATISTICAL METHODOLOGY
**→ Judgment:** YES

---
**Example 3: YES (Highly Cited)**
**Title:** "Variable selection via nonconcave penalized likelihood and its oracle properties"  
**Abstract:**  
"Variable selection is fundamental to high-dimensional statistical modeling, including nonparametric regression. Many approaches in use are stepwise selection procedures, which can be computationally expensive and ignore stochastic errors in the variable selection process. In this article, penalized likelihood approaches are proposed to handle these kinds of problems. The proposed methods select variables and estimate coefficients simultaneously. Hence they enable us to construct confidence intervals for estimated parameters. The proposed approaches are distinguished from others in that the penalty functions are symmetric, nonconcave on (0, infinity), and have singularities at the origin to produce sparse solutions. Furthermore, the penalty functions should be bounded by a constant to reduce bias and satisfy certain conditions to yield continuous solutions. A new algorithm is proposed for optimizing penalized likelihood functions. The proposed ideas are widely applicable. They are readily applied to a variety of parametric models such as generalized linear models and robust regression models. They can also be applied easily to nonparametric modeling by using wavelets and splines. Rates of convergence of the proposed penalized likelihood estimators are established. Furthermore, with proper choice of regularization parameters, we show that the proposed estimators perform as well as the oracle procedure in variable selection; namely, they work as well as if the correct submodel were known. Our simulation shows that the newly proposed methods compare favorably with other variable selection techniques. Furthermore. the standard error formulas are tested to be accurate enough for practical applications."
**Keywords:** 'hard thresholding', 'LASSO', 'nonnegative garrote', 'penalized likelihood', 'oracle estimator', 'SCAD', 'soft thresholding'
**publisher:** JOURNAL OF THE AMERICAN STATISTICAL ASSOCIATION
**→ Judgment:** YES

---
**Example 4: NO (Not Highly Cited)**
**Title:** "Profile quasi-likelihood"  
**Abstract:**  
"In this paper, the only assumptions on the distribution of data are those concerning first two moments. Our purpose is to estimate the parameter of interest in the presence of nuisance parameter under these weak assumptions on the distribution. We define a quasi-least favorable curve and construct its estimator, and then yield a profile quasi-score function of the parameter of interest, The estimator of parameter of interest obtained from this score function is asymptotically efficient. On the other hand, we employ this method to estimate the parameter in the semiparametric model. In this model the nonparametric component plays the role of nuisance parameter and it takes values in a infinite-dimensional space. The method is also available for semiparametric model and the estimator obtained by the extension is asymptotically efficient. (C) 2002 Elsevier Science B.V. All rights reserved."
**Keywords:** 'quasi likelihood', 'profile quasi-likelihood', 'efficiency', 'semiparametric'
**publisher:** STATISTICS & PROBABILITY LETTERS
**→ Judgment:** NO

---
**Example 5: NO (Not Highly Cited)**
**Title:** "Bayes methods for outliers in binomial samples"  
**Abstract:**  
"This article is concerned with the detection of outliers in a binomial sample. A Bayesian approach to the modeling of outliers is presented and examined. It is supposed that most observations are from a binomial distribution with mean pi but a small number of observations may be contaminated. That is, they are generated from a binomial sample with mean inflated (or deflated) by a factor delta. Bayes factors for the cases when either proper or improper priors are specified for pi and delta are discussed. An alternative approach, to transform the Binomial observations to approximate normality, is also presented."
**Keywords:** 'Bayesian methods', 'binomial samples', 'outliers'
**publisher:** COMMUNICATIONS IN STATISTICS-THEORY AND METHODS
**→ Judgment:** NO

---
**Example 6: NO (Not Highly Cited)**
**Title:** "Comparing treatment strategies using a synthesized clinical trial: an analysis of late versus early use of trimethoprim-sulfamethoxazole for AIDS patients"  
**Abstract:**  
"This paper applies methodology of Finkelstein and Schoenfeld [Stat. Med. 13 (1994) 1747.] to consider new treatment strategies in a synthetic clinical trial. The methodology is an approach for estimating survival functions as a composite of subdistributions defined by an auxiliary event which is intermediate to the failure. The subdistributions are usually calculated utilizing all subjects in a study, by taking into account the path determined by each individual's auxiliary event. However, the method can be used to get a composite estimate of failure from different subpopulations of patients. We utilize this application of the methodology to test a new treatment strategy, that changes therapy at later stages of disease, by combining subdistributions from different treatment arms of a clinical trial that was conducted to test therapies for prevention of pneumocystis carinii pneumonia. (C) 2001 Elsevier Science B.V. All rights reserved."
**Keywords:** 'survival', 'progression', 'semi-Markov', 'AIDS'
**publisher:** JOURNAL OF STATISTICAL PLANNING AND INFERENCE
**→ Judgment:** NO

---
**Operational Constraints**  
- Base judgment **only** on: Title, Abstract, Keywords, Publisher and Publication Year  
- Consider the **scientific landscape and prevailing trends** at the time of publication (based on the year) 
- Do **not** assume access to external databases.  

---
**Paper Information for Evaluation**  
Below is the information of the paper to be evaluated:
Title: "{title}"  
Abstract: "{abstract}"  
Keywords: "{keywords}"
Year: "{year_cleaning}"  
Publisher: "{publisher}"

---
**Required Output Format**  
Please respond with **only one** of the following:
- YES (indicating the paper is likely to become highly cited)  
- NO (indicating the paper is not likely to become highly cited)

Do not include explanations, confidence scores, or comments.
"""
    
   completion = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}]
    )
   answer = completion.choices[0].message.content
    
   return answer

In [27]:
# --- 指数退避重试（应对偶发超时/限流等） ---
def _call_with_retry(fn, max_retries=5, base_sleep=1.5, **kwargs):
    for k in range(max_retries):
        try:
            return fn(**kwargs)
        except Exception as e:
            if k == max_retries - 1:
                raise
            # 抖动 + 指数退避
            time.sleep(base_sleep * (2 ** k) * (1 + 0.25 * random.random()))

def process_with_resume_parallel(
    input_csv, output_csv, progress_txt, client,
    start_idx=0, save_step=50, max_workers=100, batch_size=1000
):
    """
    并行版：断点续跑 + 周期保存 + 线程安全
    参数：
      - save_step：每产出多少条结果就保存一次
      - max_workers：线程数（API 限流时适当调小）
      - batch_size：每次提交到线程池的任务批大小，防止一次提交太多占内存
    """
    # 1) 读数据
    df = pd.read_csv(input_csv)

    # 2) 结果列初始化
    if 'High_Citation_Label_2' not in df.columns:
        df['High_Citation_Label_2'] = None

    # 3) 计算真正的起始位置（结合 progress_txt 与 start_idx）
    resume_from = start_idx
    if os.path.exists(progress_txt):
        try:
            with open(progress_txt, 'r') as f:
                last_done = int(f.read().strip() or start_idx)
                resume_from = max(resume_from, last_done)
        except Exception:
            pass  # 读失败就忽略，用 start_idx

    # 4) 待处理索引列表（既要 >= resume_from，又要标签为空）
    mask = (df.index >= resume_from) & (df['High_Citation_Label_2'].isna())
    todo_idx = df.index[mask].tolist()

    if len(todo_idx) == 0:
        # 仍保存一次（可能有之前结果但没落盘的情况）
        df.to_csv(output_csv, index=False)
        print("No work to do. File saved to:", output_csv)
        return

    # --- 线程安全相关 ---
    write_lock = Lock()       # 写 df / 保存文件 时加锁
    save_counter = 0          # 距离上次保存累计完成的条数
    last_persist_idx = resume_from

    # --- 原业务调用封装为 worker（只传 idx，避免在线程里复制整行数据） ---
    def _worker(idx):
        row = df.loc[idx]
        # 如果你的预测函数需要 client，就在这里传入
        # 注意：把所有字段都从 df 里读，避免闭包携带大对象
        label = _call_with_retry(
            high_citation_predict,
            title=row['title'],
            abstract=row['abstract'],
            keywords=row['keywords'],
            year_cleaning=row['year_cleaning'],
            publisher=row['publisher']
        )
        return idx, label

    # --- 小工具：原子保存（先写临时文件，再替换） ---
    def _atomic_save():
        tmp_path = output_csv + ".tmp"
        df.to_csv(tmp_path, index=False)
        os.replace(tmp_path, output_csv)

    # --- 主循环：分批提交任务，as_completed 消费 ---
    pbar = tqdm(total=len(todo_idx), desc="Processing citations (parallel)")
    try:
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            for s in range(0, len(todo_idx), batch_size):
                batch = todo_idx[s:s+batch_size]
                futures = [ex.submit(_worker, idx) for idx in batch]

                for fut in as_completed(futures):
                    try:
                        idx, label = fut.result()
                    except Exception as e:
                        # 单条失败：记录、跳过
                        print(f"[WARN] Error at index {idx if 'idx' in locals() else '?'}: {e}")
                        pbar.update(1)
                        continue

                    # 写入结果（加锁）
                    with write_lock:
                        df.at[idx, 'High_Citation_Label_2'] = label
                        save_counter += 1
                        last_persist_idx = max(last_persist_idx, idx)

                        # 周期保存
                        if save_counter >= save_step:
                            _atomic_save()
                            with open(progress_txt, 'w') as f:
                                f.write(str(last_persist_idx + 1))  # 下次从下一条开始
                            save_counter = 0

                    pbar.update(1)

        # 全部结束后再做一次最终保存
        _atomic_save()
        with open(progress_txt, 'w') as f:
            f.write(str(last_persist_idx + 1))
        print("Processing complete, file saved to:", output_csv)

    finally:
        pbar.close()


In [28]:
client = OpenAI(api_key="sk-03707feaa0ac4fe8831247fa89aa5362", base_url="https://api.deepseek.com")

input_file = "output_labeled_01-05.csv"
output_file = "01-05.csv"
progress_file = "progress_01-05.txt"

try:
    with open(progress_file, 'r') as f:
        start_index = int(f.read().strip())
except:
    start_index = 0

process_with_resume_parallel(
    input_file, output_file, progress_file, client,
    start_idx=0, save_step=50, max_workers=100, batch_size=1000)

No work to do. File saved to: 01-05.csv


In [29]:
def evaluate_results(labeled_csv_path, true_label_column, pred_label_column='High_Citation_Label_2'):
    """
    评估模型预测结果的 ACC 和 TPR。
    
    :param labeled_csv_path: 已标注的输出CSV文件路径。
    :param true_label_column: CSV中存储真实标签的列名。
    :param pred_label_column: CSV中存储模型预测标签的列名。
    """
    print(f"\n--- Evaluating {os.path.basename(labeled_csv_path)} ---")
    
    try:
        df = pd.read_csv(labeled_csv_path)
        
        # 修复标签数据类型问题
        df[true_label_column] = df[true_label_column].astype(str).str.strip().replace({'1': 1, '0': 0})
        df[true_label_column] = df[true_label_column].astype(int)

        # 将预测列转为 0/1
        df[pred_label_column] = df[pred_label_column].apply(lambda x: 1 if str(x).strip().upper() == "YES" else 0)

        df_eval = df.dropna(subset=[true_label_column, pred_label_column]).copy()
        y_true = df_eval[true_label_column]
        y_pred = df_eval[pred_label_column]
        
        if len(y_true) == 0:
            print("No valid predictions to evaluate.")
            return

        # 计算 ACC (Accuracy)
        acc = accuracy_score(y_true, y_pred)
        
        # 计算 TPR (True Positive Rate), 即 Recall for the positive class (1)
        tpr = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
        
        print(f"Evaluation Metrics (on {len(y_true)} samples):")
        print(f"  Accuracy (ACC): {acc:.4f}")
        print(f"  True Positive Rate (TPR / Recall for 'YES'): {tpr:.4f}")
        
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred, target_names=['NO (0)', 'YES (1)'], zero_division=0))
        
        print("Confusion Matrix (TN, FP, FN, TP):")
        print(confusion_matrix(y_true, y_pred))
        
    except FileNotFoundError:
        print(f"Error: File not found at {labeled_csv_path}")
    except Exception as e:
        print(f"An error occurred during evaluation: {e}")

In [30]:
true_label_col = 'High_Citation_Label'

evaluate_results(labeled_csv_path=output_file,true_label_column=true_label_col)


--- Evaluating 01-05.csv ---
An error occurred during evaluation: invalid literal for int() with base 10: 'NO'
